# S1 · Mapas Hα line-to-continuum por spaxel (LSF/ghost, Xie Ec.1)

**Spec:** [`docs/plan_wavesol_stripes_2026-07-17.md`](../docs/plan_wavesol_stripes_2026-07-17.md)  |  **Bloque:** S · wavesol/stripes  |  **Run por defecto:** `ROXs12b_realigned`

Ajusta el Hα de la primaria por spaxel del halo con `phi=b(1+a·exp(−(λ−μ)²/2σ²))` (Xie+20 §4.1, Ec.1) y mapea a (line/continuo), σ, μ y P=a·b·σ·√(2π). Test de Xie Fig.3: a y σ **anticorrelados a P≈constante** ⇒ variación de LSF instrumental (alineada con slicers), no un *ghost*. Diagnóstico de apoyo a G1.

| | |
|---|---|
| **Entrada** | `cube_telcorr.fits` (realineado); geometría B3 (primaria/compañero) |
| **Salida (QC/productos)** | `stages/stageS1_qc.json`, `stages/stageS1_halpha_map.fits`, `plots/s1_halpha/`; integrado en E2 (`halpha_map_correlation` + Plot 3 del notebook E2) |
| **Consume aguas abajo** | Apoyo a la **decisión G1** (¿la variación Hα está alineada con slicers?) |


## Qué hace S1 y cómo

Por spaxel del halo (misma selección por brillo que S0) ajusta el Hα con `scipy.optimize.curve_fit` (semillas por momentos, bounds a∈[0,50], μ∈[6540,6590], σ∈[0.5,8] Å); los fits sin línea detectada (gate de S/N) devuelven NaN limpio. Reutiliza la métrica de estructura de S0 (`structure_metrics`/`stripe_profile`) sobre los mapas a y σ, con **control transversal** — la misma disciplina que S0: estructura real de slicer debe superar a su control transversal.

El núcleo (S1a) es target-agnostic y testeado (8 tests); S1b lo corre full-res sobre el realineado e integra en E2 la correlación *zonas sucias de stripes* (mapa de offset S0) vs mapas Hα.


## Cómo ejecutar de forma independiente

Diagnóstico re-ejecutable sobre un cubo existente (no re-reduce nada); rutas fijas, sin `--run-id`:

```bash
conda activate MUSE               # kernel/env con astropy + musepipe
cd MUSE-accretion-pipeline                    # raíz del repo
python -m musepipe.qc.halpha_map \
  --cube /mnt/2TB/MUSE_work/ROXs12b_realigned/cube_telcorr.fits \
  --qc-output runs/ROXs12b_realigned/stages/stageS1_qc.json \
  --map-output runs/ROXs12b_realigned/stages/stageS1_halpha_map.fits \
  --plot-output runs/ROXs12b_realigned/plots/s1_halpha/s1_realigned.png \
  --orientation vertical
python scripts/s1b_integrate_e2.py --run-dir runs/ROXs12b_realigned
```

Coste: full-res, curve_fit por spaxel ~5 min; S1b re-lee y parchea E2.

La celda `RUN=True` de más abajo hace lo mismo desde el notebook.


In [ ]:
import os, sys
# Añade notebooks/ (para _nbcommon) y la RAÍZ del repo (para importar musepipe),
# funcione el cwd en notebooks/ o en la raíz del repo.
_here = os.getcwd()
if os.path.basename(_here) != 'notebooks' and os.path.isdir(os.path.join(_here, 'notebooks')):
    _here = os.path.join(_here, 'notebooks')
for _p in (_here, os.path.dirname(_here)):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
_root = str(nb.project_root())
if _root not in sys.path:
    sys.path.insert(0, _root)   # asegura 'import musepipe'
RUN_ID = nb.resolve_run_id(None)
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))


## Ejecutar o auditar


In [ ]:
RUN = False   # -> True para RE-EJECUTAR esta etapa (regenera su QC)

if RUN:
    cmd = 'python -m musepipe.qc.halpha_map \\\n  --cube /mnt/2TB/MUSE_work/ROXs12b_realigned/cube_telcorr.fits \\\n  --qc-output runs/ROXs12b_realigned/stages/stageS1_qc.json \\\n  --map-output runs/ROXs12b_realigned/stages/stageS1_halpha_map.fits \\\n  --plot-output runs/ROXs12b_realigned/plots/s1_halpha/s1_realigned.png \\\n  --orientation vertical\npython scripts/s1b_integrate_e2.py --run-dir runs/ROXs12b_realigned'.replace('$RUN', RUN_ID)
    print('ejecutando:', cmd)
    import subprocess
    subprocess.run(cmd, shell=True, cwd=str(nb.project_root()), check=True)
else:
    print('Modo auditoría (RUN=False): se carga el QC existente abajo.')


## QC / resultados


In [ ]:
qc = nb.load_qc('stages/stageS1_qc.json', RUN_ID)
nb.show(qc, keys=['halpha_map.sigma_median_A', 'halpha_map.a_structure_significance', 'halpha_map.a_transverse_significance', 'halpha_map.sigma_structure_significance', 'halpha_map.sigma_transverse_significance', 'halpha_map.corr_a_sigma', 'halpha_map.P_cov', 'halpha_map.n_fit'], title='S1')


## Resultado (realineado, full-res)

n_fit=29203, σ_median=2.01 Å. **NO alineado con slicers:** a struct 6.7× vs transversal 8.9×; σ struct 8.6× vs transversal 7.9× (estructura ≤ control ⇒ isótropo/radial, núcleo-vs-halo). corr(a,σ)=−0.57 pero **P_cov=1.07** (P no constante) ⇒ NO es el caso instrumental-LSF de Xie Fig.3. La correlación por columna stripe↔σ (−0.91) es un confundido radial (control transversal −0.905). Consistente con G1: el cubo combinado es ciego a stripes.


In [ ]:
q = nb.load_qc('stages/stageS1_qc.json', RUN_ID)['halpha_map']
print(f"n_fit={q['n_fit']}  sigma_median={q['sigma_median_A']:.3f} A")
print(f"a:     struct {q['a_structure_significance']:.1f}x  transv {q['a_transverse_significance']:.1f}x")
print(f"sigma: struct {q['sigma_structure_significance']:.1f}x  transv {q['sigma_transverse_significance']:.1f}x")
print(f"corr(a,sigma)={q['corr_a_sigma']:.2f}  P_cov={q['P_cov']:.2f} (Xie Fig.3 pide P~const)")
aligned = lambda s,t: s>3 and s>2*t
print('a slicer-aligned:', aligned(q['a_structure_significance'], q['a_transverse_significance']),
      '| sigma slicer-aligned:', aligned(q['sigma_structure_significance'], q['sigma_transverse_significance']))


## Mapas S1 (a, σ, P, y a-vs-σ)


In [ ]:
from IPython.display import Image, display
p = nb.run_dir(RUN_ID) / 'plots' / 's1_halpha' / 's1_realigned.png'
if p.exists(): display(Image(filename=str(p)))
else: print('falta', p, '- corre la etapa (arriba).')


## Decisiones y notas
- Núcleo S1a target-agnostic + 8 tests; S1b integra en E2 (aditivo) con control transversal. · [`docs/plan_wavesol_stripes_pasos_agente.md`](../docs/plan_wavesol_stripes_pasos_agente.md)
- a/σ NO alineados con slicers (radial); apoya el cierre G1 (sin stripes en el combinado). · [`docs/decision_g1_wavesol_2026-07-17.md`](../docs/decision_g1_wavesol_2026-07-17.md)


## Checks


In [ ]:
q = nb.load_qc('stages/stageS1_qc.json', RUN_ID)['halpha_map']
print('sigma_median_A =', round(q['sigma_median_A'],3))
print('corr_a_sigma   =', round(q['corr_a_sigma'],3), '(P_cov', round(q['P_cov'],2),')')


## Conclusión (registrada, 2026-07-18)

Los mapas Hα a/σ del cubo combinado **no muestran estructura de slicer** (estructura ≤ control transversal) y **no** cumplen la firma instrumental-LSF de Xie Fig.3 (P no constante). Es diagnóstico de apoyo al **cierre G1**: el combinado es ciego a los stripes (confirmado por S0 por-exposición). Sin interpretación ghost-vs-instrumental adicional (se resolvió por-exposición).
